In [1]:
!pip install -q faiss-cpu transformers sentencepiece \
    torch torchvision Pillow tqdm scenedetect[opencv] \
    faster-whisper openai pandas

from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

!kaggle datasets download -d don121/scene-detect-cu-itmo

! unzip "scene-detect-cu-itmo.zip" >> /dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/don121/scene-detect-cu-itmo
License(s): unknown
scene-detect-cu-itmo.zip: Skipping, found more recently modified local copy (use --force to force download)
replace __huggingface_repos__.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [7]:
import sys
import os

CODE_DIR = "/content/drive/MyDrive/cu_itmo_hack/ai-chemp-cu-itmo"     # репозиторий с кодом

WORK_DIR = "/content"

OUTPUT_DIR = os.path.join(WORK_DIR, "output")   # keyframes, all_scenes.json
DB_DIR     = os.path.join(WORK_DIR, "db")       # FAISS индексы
QUESTIONS_PATH = os.path.join(CODE_DIR, "eval/eval_dataset.json")

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Code:   {CODE_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"DB:     {DB_DIR}")
print(f"Device: {DEVICE}")

Code:   /content/drive/MyDrive/cu_itmo_hack/ai-chemp-cu-itmo
Output: /content/output
DB:     /content/db
Device: cpu


## 1. Построение индекса: Scene Detection → ASR → SigLIP → FAISS

In [3]:
# from build_vectordb import build_db

# # Если есть готовый all_scenes.json (+ keyframes) — укажи путь, scene detect пропустится
# SCENES_JSON_PATH = None  # например: "/kaggle/input/video-rag-scenes/all_scenes.json"

# build_db(
#     data_dir=DATA_DIR,
#     output_dir=OUTPUT_DIR,
#     db_dir=DB_DIR,
#     device=DEVICE,
#     whisper_model="base",
#     scenes_json_path=SCENES_JSON_PATH,
# )

## 2. Загрузка построенного индекса и модели

In [4]:
from src.embed import SIGLIP_MODEL, load_db
from transformers import AutoProcessor, AutoModel
import pandas as pd

# Загружаем FAISS индексы и метадату из только что построенной БД
print("Loading FAISS database...")
vis_idx, txt_idx, metadata = load_db(DB_DIR)
print(f"  {vis_idx.ntotal} scenes in index")
print(f"  Видео: {set(m['video'] for m in metadata)}")

# Загружаем SigLIP для encode_query при evaluation
print(f"\nLoading SigLIP ({SIGLIP_MODEL})...")
proc = AutoProcessor.from_pretrained(SIGLIP_MODEL)
model = AutoModel.from_pretrained(SIGLIP_MODEL).to(DEVICE).eval()
print("Ready!")

Loading FAISS database...
  3037 scenes in index
  Видео: {'shrek.mp4', 'brat.mp4', 'poimai_menya_esli_smozeh.mp4'}

Loading SigLIP (google/siglip-base-patch16-224)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Ready!


## 3. Обзор индекса — что внутри

In [5]:
def fmt_time(sec):
    m, s = divmod(int(sec), 60)
    return f"{m:02d}:{s:02d}"

df_meta = pd.DataFrame(metadata)
df_meta["duration"] = df_meta["end_sec"] - df_meta["start_sec"]
df_meta["time"] = df_meta.apply(lambda r: f"{fmt_time(r['start_sec'])}–{fmt_time(r['end_sec'])}", axis=1)
df_meta["has_transcript"] = df_meta["transcript"].apply(lambda t: bool(t and t.strip()))

print(f"Всего сцен: {len(df_meta)}")
print(f"С транскриптом: {df_meta['has_transcript'].sum()} / {len(df_meta)}")
print(f"Средняя длина сцены: {df_meta['duration'].mean():.1f}s")
print(f"Мин/макс: {df_meta['duration'].min():.1f}s / {df_meta['duration'].max():.1f}s")

df_meta[["faiss_id", "video", "time", "duration", "has_transcript", "transcript"]].head(10)

Всего сцен: 3037
С транскриптом: 1801 / 3037
Средняя длина сцены: 6.4s
Мин/макс: 0.6s / 418.6s


,faiss_id,video,time,duration,has_transcript,transcript
0,0,brat.mp4,00:00–00:01,1.72,False,
1,1,brat.mp4,00:01–00:02,1.04,False,
2,2,brat.mp4,00:02–00:17,14.64,True,ЗВОНОК В ДВЕРЬ
3,3,brat.mp4,00:17–00:21,4.56,False,
4,4,brat.mp4,00:21–00:26,4.44,False,
5,5,brat.mp4,00:26–00:58,32.56,False,
6,6,brat.mp4,00:58–01:05,6.44,False,
7,7,brat.mp4,01:05–01:16,11.16,False,
8,8,brat.mp4,01:16–01:20,3.72,False,
9,9,brat.mp4,01:20–01:24,4.60,False,


## 4. Загрузка eval-датасета

In [8]:
from eval.dataset import EvalDataset

dataset = EvalDataset.load(QUESTIONS_PATH)
print(dataset)

q_rows = []
for e in dataset.entries:
    spans = ""
    if e.time_spans:
        spans = ", ".join(f"{fmt_time(s.start)}–{fmt_time(s.end)}" for s in e.time_spans)
    q_rows.append({
        "query": e.query,
        "type": e.query_type,
        "video": e.video_id,
        "GT time": spans,
    })
pd.DataFrame(q_rows)

EvalDataset({'total': 18, 'by_type': {'visual': 18}, 'by_video': {'shrek.mp4': 10, 'brat.mp4': 8}, 'with_time_spans': 18, 'with_expected_answer': 0})


,query,type,video,GT time
0,момент когда шрек спасает осла от дракона,visual,shrek.mp4,38:00–40:00
1,шрек и осел смотрят на луну,visual,shrek.mp4,48:00–49:00
2,фиона превращается в монстра,visual,shrek.mp4,61:42–62:42
3,поход по мосту над лавой,visual,shrek.mp4,30:00–31:00
4,шрек стоит с подсолнухом,visual,shrek.mp4,64:00–65:00
5,фиона с шреком едят рагу,visual,shrek.mp4,58:00–59:00
6,гг пришли на парковку сера ланселота,visual,shrek.mp4,20:00–21:00
7,гг пришли на парковку сэра ланселота,visual,shrek.mp4,20:00–21:00
8,шрек надувает лягушку,visual,shrek.mp4,56:20–57:20
9,турнир сэра ланселота,visual,shrek.mp4,23:30–24:30


## 5. Запуск evaluation

In [9]:
import logging
from eval.evaluate import Evaluator

# DEBUG — видно HIT/MISS и детали для каждого запроса
logging.basicConfig(level=logging.DEBUG, format="%(message)s", force=True)

TOP_K = 5
OVERLAP_THRESHOLD = 0.3

evaluator = Evaluator(
    vis_index=vis_idx,
    txt_index=txt_idx,
    metadata=metadata,
    model=model,
    processor=proc,
    device=DEVICE,
    top_k=TOP_K,
)

results = evaluator.evaluate(
    dataset,
    top_k=TOP_K,
    overlap_threshold=OVERLAP_THRESHOLD,
    judge=False,
)

print(f"\n{'='*60}")
print(results)
print(f"{'='*60}")

[MISS] query='момент когда шрек спасает осла от дракона'
  GT:        scene 2362 [37:59–38:00], scene 2363 [38:00–38:01], scene 2364 [38:01–38:03], scene 2365 [38:03–38:06], scene 2366 [38:06–38:08], scene 2367 [38:08–38:10], scene 2368 [38:10–38:12], scene 2369 [38:12–38:15], scene 2370 [38:15–38:16], scene 2371 [38:16–38:17], scene 2372 [38:17–38:19], scene 2373 [38:19–38:20], scene 2374 [38:20–38:25], scene 2375 [38:25–38:27], scene 2376 [38:27–38:28], scene 2377 [38:28–38:29], scene 2378 [38:29–38:33], scene 2379 [38:33–38:35], scene 2380 [38:35–38:36], scene 2381 [38:36–38:39], scene 2382 [38:39–38:40], scene 2383 [38:40–38:42], scene 2384 [38:42–38:44], scene 2385 [38:44–38:45], scene 2386 [38:45–38:49], scene 2387 [38:49–38:50], scene 2388 [38:50–38:52], scene 2389 [38:52–38:54], scene 2390 [38:54–38:56], scene 2391 [38:56–38:57], scene 2392 [38:57–38:58], scene 2393 [38:58–39:00], scene 2394 [39:00–39:06], scene 2395 [39:06–39:19], scene 2396 [39:19–39:21], scene 2397 [39:21–39


Retrieval: Recall@K=0.024  MRR=0.169  nDCG=0.041  tIoU=0.025  offset=3:45  (n=15)


## 6. Per-query результаты

In [10]:
if results.retrieval:
    df = pd.DataFrame(results.retrieval.per_query)
    df = df.sort_values(["recall", "tiou"], ascending=[True, True])

    display_cols = ["query", "hit", "recall", "rr", "ndcg", "tiou", "offset"]
    styled = df[display_cols].style.format({
        "recall": "{:.2f}",
        "rr": "{:.2f}",
        "ndcg": "{:.2f}",
        "tiou": "{:.3f}",
    }).map(
        lambda v: "background-color: #ffcccc" if v == "MISS" else "",
        subset=["hit"]
    )
    display(styled)
else:
    print("Нет retrieval-метрик (нет вопросов с time_spans)")

,query,hit,recall,rr,ndcg,tiou,offset
0,момент когда шрек спасает осла от дракона,MISS,0.00,0.00,0.00,0.000,6:34
1,шрек и осел смотрят на луну,HIT,0.00,0.00,0.00,0.000,0:25
2,фиона превращается в монстра,MISS,0.00,0.00,0.00,0.000,4:14
4,шрек стоит с подсолнухом,HIT,0.00,0.00,0.00,0.000,0:08
5,фиона с шреком едят рагу,MISS,0.00,0.00,0.00,0.000,1:30
9,турнир сэра ланселота,HIT,0.00,0.00,0.00,0.000,0:08
10,крылатая фраза бездомного по отношению к Даниле после того как он спас его поле ограбления на улеченом рынке,MISS,0.00,0.00,0.00,0.000,4:57
11,Данила убегае от бандитов поле убийства на рынке,HIT,0.00,0.00,0.00,0.000,0:34
12,Цвет трамвая на котором данила убегает от бандитов после убийства на рынке,MISS,0.00,0.00,0.00,0.000,26:43
13,"Разговор Данилы и Немца, когда Данила ранен",MISS,0.00,0.00,0.00,0.000,8:48


## 7. Детали конкретного запроса

In [12]:
QUERY_IDX = 0  # поменяй индекс чтобы посмотреть другой запрос

if results.retrieval and QUERY_IDX < len(results.retrieval.per_query):
    pq = results.retrieval.per_query[QUERY_IDX]
    print(f"Query: {pq['query']}")
    print(f"Hit: {pq['hit']}  Recall={pq['recall']:.2f}  MRR={pq['rr']:.2f}  "
          f"tIoU={pq['tiou']:.3f}  offset={pq['offset']}")

    print(f"\nGround truth:")
    for gt in pq["ground_truth_spans"]:
        fid = gt["faiss_id"]
        transcript = metadata[fid].get("transcript", "")[:80]
        print(f"  scene {fid} [{gt['time']}] | {transcript}")

    print(f"\nRetrieved:")
    for ret in pq["retrieved_details"]:
        fid = ret["faiss_id"]
        marker = " ← HIT" if fid in pq["ground_truth"] else ""
        transcript = metadata[fid].get("transcript", "")[:80]
        print(f"  scene {fid} [{ret['time']}] score={ret['score']:.4f}{marker}")
        print(f"    transcript: {transcript}")

Query: момент когда шрек спасает осла от дракона
Hit: MISS  Recall=0.00  MRR=0.00  tIoU=0.000  offset=6:34

Ground truth:
  scene 2362 [37:59–38:00] | Я по лоту броняне.
  scene 2363 [38:00–38:01] | 
  scene 2364 [38:01–38:03] | Я предправду, задержаться, но ты сама понимаешь.
  scene 2365 [38:03–38:06] | 
  scene 2366 [38:06–38:08] | Эй, не в ступы. Это мой нечли хост.
  scene 2367 [38:08–38:10] | Смотрим, ты разберешь силы, оторвешь.
  scene 2368 [38:10–38:12] | 
  scene 2369 [38:12–38:15] | Что ты тогда будем делать? Нет, не надо, не надо.
  scene 2370 [38:15–38:16] | 
  scene 2371 [38:16–38:17] | Я спрятнешь, спрятнешь, спрятнешь.
  scene 2372 [38:17–38:19] | 
  scene 2373 [38:19–38:20] | 
  scene 2374 [38:20–38:25] | 
  scene 2375 [38:25–38:27] | 
  scene 2376 [38:27–38:28] | 
  scene 2377 [38:28–38:29] | 
  scene 2378 [38:29–38:33] | 
  scene 2379 [38:33–38:35] | 
  scene 2380 [38:35–38:36] | 
  scene 2381 [38:36–38:39] | 
  scene 2382 [38:39–38:40] | 
  scene 2383 [38:40–38:42] 

## 8. Сравнение top_k

In [13]:
# отключаем debug-логи для чистоты
logging.getLogger().setLevel(logging.WARNING)

k_values = [1, 3, 5, 10, 15]
rows = []

for k in k_values:
    r = evaluator.evaluate(dataset, top_k=k, overlap_threshold=OVERLAP_THRESHOLD, judge=False)
    if r.retrieval:
        rows.append({
            "top_k": k,
            "Recall@K": r.retrieval.recall_at_k,
            "MRR": r.retrieval.mrr,
            "nDCG": r.retrieval.ndcg,
            "tIoU": r.retrieval.mean_tiou,
            "offset_sec": r.retrieval.mean_offset_sec,
        })

if rows:
    df_k = pd.DataFrame(rows)
    display(df_k.style.format({
        "Recall@K": "{:.3f}",
        "MRR": "{:.3f}",
        "nDCG": "{:.3f}",
        "tIoU": "{:.3f}",
        "offset_sec": "{:.1f}",
    }))

,top_k,Recall@K,MRR,nDCG,tIoU,offset_sec
0,1,0.004,0.067,0.011,0.009,1877.3
1,3,0.017,0.156,0.035,0.021,638.7
2,5,0.024,0.169,0.041,0.025,225.8
3,10,0.045,0.186,0.055,0.035,104.4
4,15,0.069,0.207,0.072,0.046,51.9


## 9. Ручной поиск — проверка конкретного запроса

In [14]:
from search import encode_query, search_index, reciprocal_rank_fusion
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage, Markdown


def manual_search(query, top_k=5):
    """Ручной поиск с визуализацией найденных keyframe."""
    q_vec = encode_query(query, model, proc, DEVICE)

    n_search = min(top_k * 3, vis_idx.ntotal)
    _, vis_ids = search_index(vis_idx, q_vec, n_search)
    _, txt_ids = search_index(txt_idx, q_vec, n_search)
    hits = reciprocal_rank_fusion([vis_ids.tolist(), txt_ids.tolist()])

    display(Markdown(f"### Запрос: *{query}*"))

    for rank, (score, fid) in enumerate(hits[:top_k], 1):
        scene = metadata[int(fid)]
        time_str = f"{fmt_time(scene['start_sec'])}–{fmt_time(scene['end_sec'])}"
        transcript = scene.get("transcript", "") or "(нет речи)"

        display(Markdown(
            f"**#{rank}** [{time_str}] score={score:.4f}\n\n"
            f"> {transcript[:150]}"
        ))

        kf = scene.get("keyframe_path", "")
        if kf and os.path.exists(kf):
            display(IPImage(filename=kf, width=480))


manual_search("зелёный персонаж бежит по болоту")

### Запрос: *зелёный персонаж бежит по болоту*

**#1** [44:46–44:50] score=0.0299

> Но в лесу водятся разбойники.

**#2** [27:52–27:58] score=0.0164

> Разве кто-нибудь когда-нибудь скажет, нетерплю, слоёное мороженое. Порфе. Порфе-то лучше. Нет.

**#3** [14:59–15:04] score=0.0164

> Бежать!

**#4** [55:58–55:59] score=0.0161

> (нет речи)

**#5** [08:16–08:21] score=0.0161

> Ты зеленый, большой, как танк. Будете наводить уже с навстречных.

## 10. Сохранение результатов

In [15]:
import json

OUTPUT_PATH = "/content/eval_results.json"
# OUTPUT_PATH = "../eval/results.json"  # для локального запуска

out = {"config": {"top_k": TOP_K, "overlap_threshold": OVERLAP_THRESHOLD}}

if results.retrieval:
    out["retrieval"] = {
        "recall_at_k": results.retrieval.recall_at_k,
        "mrr": results.retrieval.mrr,
        "ndcg": results.retrieval.ndcg,
        "mean_tiou": results.retrieval.mean_tiou,
        "mean_offset_sec": results.retrieval.mean_offset_sec,
        "total_queries": results.retrieval.total_queries,
        "per_query": results.retrieval.per_query,
    }

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}")

Saved to /content/eval_results.json
